# 02 — DataFrames, I/O, and Schemas

Creating DataFrames, reading/writing the common file formats, managing schemas explicitly, and the UDF-vs-built-in-function tradeoff that comes up in almost every Spark interview.

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark — read it as a reference and run
> cell-by-cell once your environment is set up.

## 1. Creating DataFrames

In practice you almost always create DataFrames by **reading data**, but knowing how to build one from Python objects is useful for tests and demos.

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# From a list of tuples + explicit column names
df1 = spark.createDataFrame(
    [("alice", 30, 95000.0), ("bob", 45, 120000.0), ("cara", 27, 88000.0)],
    schema=["name", "age", "salary"],
)

# From Row objects (mirrors what .collect() returns)
df2 = spark.createDataFrame([Row(name="dan", age=51, salary=140000.0)])

df1.union(df2).show()

## 2. Schemas: infer vs explicit — and why interviewers ask about it

`inferSchema=True` makes Spark do a **full extra pass** over the data (for CSV/JSON) just to guess types — costly on large files, and it can guess wrong (e.g. a zip-code column full of leading zeros becomes `IntegerType` and silently loses the zeros).

**Always define an explicit `StructType` schema in production pipelines**: it's faster (single pass), fails fast on unexpected data instead of silently coercing it, and documents the contract of the dataset.

In [ ]:
explicit_schema = StructType([
    StructField("user_id", IntegerType(), nullable=False),
    StructField("event_name", StringType(), nullable=False),
    StructField("amount", DoubleType(), nullable=True),
])

# df = spark.read.schema(explicit_schema).csv("events.csv", header=True)
# df = spark.read.schema(explicit_schema).json("events.json")
# Parquet/ORC/Avro carry their own schema in the file footer — no inference needed,
# which is one reason they're preferred for pipeline-to-pipeline handoffs.
# df = spark.read.parquet("events.parquet")

## 3. Format comparison — a classic "which format would you use" question

| Format | Schema-on-read | Splittable | Columnar | Compression | Typical use |
|---|---|---|---|---|---|
| CSV | no (inferred/loose) | yes | no | none/gzip (not splittable) | raw ingestion, human-editable |
| JSON | no (inferred/loose) | yes (line-delimited) | no | gzip | semi-structured/nested events |
| Parquet | yes (embedded) | yes | **yes** | snappy/gzip, per-column | analytics, the default for lakes |
| ORC | yes (embedded) | yes | yes | zlib/snappy | Hive-heavy stacks |
| Avro | yes (embedded) | yes | no (row-based) | deflate/snappy | streaming/Kafka, schema evolution |

**Rule of thumb:** land raw data as JSON/CSV, transform once, and persist curated tables as **Parquet** — columnar storage lets Spark skip whole column chunks it doesn't need (column pruning) and skip whole row groups via min/max statistics (predicate pushdown).

## 4. Column operations

`select`, `withColumn`, `filter`/`where`, `when/otherwise`, casting — the bread-and-butter API used in every transformation step.

In [ ]:
from pyspark.sql.functions import col, when, lit

salaries = df1.union(df2)

result = (
    salaries
    .withColumn("seniority", when(col("age") >= 45, lit("senior")).otherwise(lit("mid")))
    .withColumn("salary_k", (col("salary") / 1000).cast("int"))
    .select("name", "seniority", "salary_k")
    .filter(col("salary_k") > 90)
)
result.show()

## 5. UDFs vs built-in functions — a common interview trap

- **Built-in `pyspark.sql.functions`** run inside the JVM, fully visible to Catalyst/Tungsten — fast, vectorized, optimizable. Always prefer these.
- **Python UDFs** (`@udf`) serialize every row from the JVM to a Python worker process, run your Python function one row at a time, and serialize the result back. This row-by-row Python↔JVM hop is slow and is a Catalyst black box (no optimization inside the UDF).
- **Pandas UDFs** (`@pandas_udf`, vectorized via Apache Arrow) batch many rows into a pandas Series per call — far less serialization overhead than row-at-a-time UDFs, though still slower than pure built-ins.

**Interview answer:** "Always check `pyspark.sql.functions` first; reach for a pandas UDF (vectorized) before a plain Python UDF; avoid Python UDFs on hot paths / huge datasets."

In [ ]:
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType

# Slow path: row-at-a-time Python UDF
@udf(returnType=StringType())
def classify_py(age):
    return "senior" if age >= 45 else "mid"

# Faster path: vectorized pandas UDF (operates on a pandas Series batch)
@pandas_udf(StringType())
def classify_pandas(age_series):
    return age_series.apply(lambda a: "senior" if a >= 45 else "mid")

# Fastest / preferred: pure built-in expression, no UDF at all
# .withColumn("seniority", when(col("age") >= 45, "senior").otherwise("mid"))

salaries.withColumn("seniority", classify_pandas(col("age"))).show()

## 6. Interview Q&A

1. **"Your CSV read is slow — why?"** — `inferSchema=True` forces a full extra scan of the file to guess types; pass an explicit `StructType` schema.
2. **"Why is Parquet preferred over CSV for a data lake?"** — columnar layout + embedded schema + statistics enable column pruning and predicate pushdown, and it's splittable and compresses far better.
3. **"When would you still use a Python UDF despite the cost?"** — when the logic genuinely can't be expressed with built-ins/pandas UDFs (e.g. calling an external library per row) and the dataset is small enough that the overhead doesn't matter.
4. **"What does `nullable=False` in a schema actually enforce?"** — it's a hint/contract, not a hard runtime constraint on read for most sources; Spark does not always validate it — don't rely on it for data quality guarantees, use explicit `.filter(col(x).isNotNull())` / validation checks.

## Summary

- Define explicit schemas in production; avoid `inferSchema` on large files.
- Parquet is the default choice for curated/analytics data.
- Prefer built-in functions > pandas UDFs > Python UDFs, in that order.
- Next: `03_joins_and_broadcast.ipynb`.